# 🤖 Pelatihan Model ML — Klasifikasi GitHub Issues .NET
### Input: `so_dotnet_dataset_scraped.csv` (hasil dari `scraping_github_dotnet.ipynb`)

**Alur notebook ini:**
1. Upload `so_dotnet_dataset_scraped.csv` yang dihasilkan notebook scraping
2. Load & validasi data dari CSV tersebut
3. Pelabelan 3 kelas dari kolom CSV scraping
4. Ekstraksi fitur dari kolom CSV scraping
5. 3 skema pelatihan ML
6. Evaluasi & inference

| Skema | Algoritma | Fitur | Split |
|---|---|---|---|
| 1 | Logistic Regression (C=1.5) | TF-IDF + Numerik + Label GitHub | 80/20 |
| 2 | LinearSVC (C=1.0) | Semua Fitur | 80/20 |
| 3 | Logistic Regression (C=3.0) | Semua Fitur | 70/30 |


## 📦 Cell 1 — Install & Import

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn openpyxl --quiet

import os, re, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
from scipy.sparse import hstack, csr_matrix
from IPython.display import display

warnings.filterwarnings("ignore")
os.makedirs("/content/model_output", exist_ok=True)

print("✅ Semua library siap!")


## 📂 Cell 2 — Upload File CSV Hasil Scraping

> **WAJIB:** Upload file **`so_dotnet_dataset_scraped.csv`**  
> File ini dihasilkan oleh `scraping_github_dotnet.ipynb`  
> Jangan upload file lain — nama file harus `so_dotnet_dataset_scraped.csv`


In [ ]:
from google.colab import files

# ── Nama file CSV yang WAJIB diupload ──────────────────────────
CSV_FILENAME = "so_dotnet_dataset_scraped.csv"

print(f"📤 Upload file hasil scraping: {CSV_FILENAME}")
print("   (jalankan scraping_github_dotnet.ipynb terlebih dahulu)")
print()
uploaded = files.upload()

# Validasi nama file yang diupload
uploaded_name = list(uploaded.keys())[0]
if uploaded_name != CSV_FILENAME:
    print(f"⚠️  Peringatan: file yang diupload '{uploaded_name}'")
    print(f"   bukan '{CSV_FILENAME}'")
    print(f"   Lanjutkan dengan file yang diupload...")
    CSV_FILENAME = uploaded_name

print(f"\n✅ File diterima     : {CSV_FILENAME}")
print(f"   Ukuran            : {len(uploaded[uploaded_name])/1024:.1f} KB")


## 🔍 Cell 3 — Load & Validasi CSV Hasil Scraping

In [ ]:
# ── Load CSV hasil scraping ────────────────────────────────────
df = pd.read_csv(CSV_FILENAME)

# Bersihkan kolom index jika ada
for col in ["no","Unnamed: 0"]:
    if col in df.columns:
        df = df.drop(columns=[col])
df = df.reset_index(drop=True)

# ── Validasi kolom — semua kolom ini ada di CSV hasil scraping ──
KOLOM_SCRAPING = [
    "id",             # ID unik issue dari GitHub
    "judul",          # Judul issue (title)
    "topik",          # Topik/kategori dari query scraping
    "labels_github",  # Label-label GitHub issue
    "state",          # Status: open / closed
    "komentar",       # Jumlah komentar
    "reactions",      # Jumlah reaksi
    "body_clean",     # Body issue yang sudah dibersihkan
    "panjang_judul",  # Panjang karakter judul
    "panjang_body",   # Panjang karakter body
    "ada_code_block", # 1 jika ada code block, 0 jika tidak
    "tanggal_dibuat", # Tanggal issue dibuat
    "ditutup",        # 1 jika closed, 0 jika open
    "url",            # URL issue di GitHub
    "author",         # Username pembuat issue
]

missing = [k for k in KOLOM_SCRAPING if k not in df.columns]
print("=" * 60)
print(f"  DATASET DIMUAT: {CSV_FILENAME}")
print("=" * 60)
print(f"  Total baris  : {len(df):,}")
print(f"  Total kolom  : {len(df.columns)}")
print()
if missing:
    print(f"⚠️  Kolom tidak ditemukan: {missing}")
else:
    print(f"✅ Semua {len(KOLOM_SCRAPING)} kolom scraping tersedia")

print("\n🔎 5 baris pertama CSV hasil scraping:")
display(df[["id","judul","topik","state","komentar","ditutup","url"]].head())


## 🧹 Cell 4 — Cleaning Data

In [ ]:
df_clean = df.copy()
df_clean = df_clean.dropna(subset=["judul"])
df_clean["judul"]         = df_clean["judul"].astype(str).str.strip()
df_clean["body_clean"]    = df_clean["body_clean"].fillna("").astype(str)
df_clean["labels_github"] = df_clean["labels_github"].fillna("").astype(str)

for col in ["komentar","reactions","panjang_judul",
            "panjang_body","ada_code_block","ditutup"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0).astype(int)

df_clean = df_clean.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"Sebelum cleaning : {len(df):,} baris")
print(f"Setelah cleaning : {len(df_clean):,} baris")
print(f"Baris terhapus   : {len(df) - len(df_clean)}")
print()
print("Distribusi per topik (dari kolom 'topik' CSV scraping):")
display(df_clean["topik"].value_counts().to_frame("jumlah_issue"))


## 🏷️ Cell 5 — Pelabelan 3 Kelas

Pelabelan menggunakan kolom dari CSV hasil scraping:  
`ditutup` · `komentar` · `labels_github`

| Label | Kondisi (dari kolom CSV scraping) |
|---|---|
| 🟢 `positif` | `ditutup=1` AND (`komentar≥3` OR label done/feature) |
| 🔴 `negatif` | `ditutup=0` AND `komentar=0` |
| 🟡 `netral` | kondisi lainnya |


In [ ]:
def assign_label(row):
    """
    Labeling 3 kelas menggunakan kolom dari CSV hasil scraping:
      - row['ditutup']       : kolom dari scraping (0/1)
      - row['komentar']      : kolom dari scraping (integer)
      - row['labels_github'] : kolom dari scraping (string)
    """
    ditutup = int(row["ditutup"]) == 1       # kolom CSV scraping
    kom     = int(row["komentar"])            # kolom CSV scraping
    lbl_gh  = str(row["labels_github"]).lower()  # kolom CSV scraping

    done_kw = any(k in lbl_gh for k in
                  ["fixed","resolved","done","closed","wontfix","duplicate"])
    feat_kw = any(k in lbl_gh for k in
                  ["enhancement","feature","improvement","request"])

    if ditutup and (kom >= 3 or done_kw or feat_kw):
        return "positif"
    elif not ditutup and kom == 0:
        return "negatif"
    else:
        return "netral"

# Terapkan ke data hasil scraping
df_clean["label"] = df_clean.apply(assign_label, axis=1)

colors_map = {"positif":"#2ecc71","netral":"#f39c12","negatif":"#e74c3c"}
print("=" * 50)
print(f"  DISTRIBUSI LABEL — dari {CSV_FILENAME}")
print("=" * 50)
for lbl, cnt in df_clean["label"].value_counts().items():
    bar = "█" * int(cnt / len(df_clean) * 40)
    print(f"  {lbl:<10}: {cnt:>5,}  ({cnt/len(df_clean)*100:5.1f}%)  {bar}")
print(f"\n  Total: {len(df_clean):,} sampel | 3 kelas")

# Visualisasi distribusi label & topik
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"Dataset: {CSV_FILENAME} ({len(df_clean):,} rows)",
             fontweight="bold")
lv = df_clean["label"].value_counts()
axes[0].pie(lv.values, labels=lv.index, autopct="%1.1f%%",
            colors=[colors_map.get(l,"gray") for l in lv.index],
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[0].set_title("Distribusi Label (3 Kelas)", fontweight="bold")

tv = df_clean["topik"].value_counts()
axes[1].barh(tv.index, tv.values, color="steelblue", edgecolor="white")
axes[1].set_title("Distribusi Topik Hasil Scraping", fontweight="bold")
axes[1].set_xlabel("Jumlah"); axes[1].invert_yaxis()
for i,v in enumerate(tv.values):
    axes[1].text(v+1,i,str(v),va="center",fontsize=9)
plt.tight_layout(); plt.show()

# Simpan CSV berlabel
LABELED_CSV = "/content/model_output/dataset_labeled.csv"
df_clean.to_csv(LABELED_CSV, index=False)
print(f"\n💾 Dataset berlabel disimpan: {LABELED_CSV}")


## 🔧 Cell 6 — Ekstraksi Fitur

Semua fitur diambil **langsung dari kolom CSV hasil scraping**:

| Fitur | Kolom CSV sumber |
|---|---|
| TF-IDF | `judul` + `body_clean` |
| Numerik | `komentar`, `reactions`, `panjang_judul`, `panjang_body`, `ada_code_block`, `ditutup` |
| Label GitHub | `labels_github` |
| Topik | `topik` |


## 🔧 Cell 6 — Preprocessing Teks & Fitur Non-Teks

> **Anti-leakage:** TF-IDF **belum** di-fit di sini.  
> Fit TF-IDF dilakukan **per skema**, hanya pada data train masing-masing split.


In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder

def clean_text(t):
    """Bersihkan teks dari kolom judul dan body_clean CSV scraping."""
    t = str(t).lower()
    t = re.sub(r"http\S+", "", t)
    t = re.sub(r"[^a-z0-9\s#.+\-_]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# ── Teks gabungan: judul (3x bobot) + body_clean ─────────────
# Sumber: kolom 'judul' dan 'body_clean' dari CSV scraping
df_clean["text_input"] = (
    df_clean["judul"].apply(clean_text) + " " +
    df_clean["judul"].apply(clean_text) + " " +
    df_clean["judul"].apply(clean_text) + " " +
    df_clean["body_clean"].apply(clean_text)
)

# ── Fitur label GitHub (one-hot) ──────────────────────────────
# Sumber: kolom 'labels_github' dari CSV scraping
def github_label_features(lbl_str):
    lbl = str(lbl_str).lower()
    return [
        int(any(k in lbl for k in ["bug","error","crash","fail"])),
        int(any(k in lbl for k in ["enhancement","feature","improvement"])),
        int(any(k in lbl for k in ["fixed","resolved","done","closed"])),
        int("duplicate" in lbl),
        int("wontfix" in lbl),
        int(any(k in lbl for k in ["invalid","spam"])),
    ]

X_lbl = csr_matrix(np.array(
    df_clean["labels_github"].apply(github_label_features).tolist(),
    dtype=float))

# ── Fitur numerik (log-transform) ────────────────────────────
# Sumber: kolom komentar, reactions, panjang_judul,
#         panjang_body, ada_code_block, ditutup dari CSV scraping
df_clean["log_komentar"] = np.log1p(df_clean["komentar"])
df_clean["log_body"]     = np.log1p(df_clean["panjang_body"])
df_clean["engagement"]   = np.log1p(df_clean["komentar"] + df_clean["reactions"])

NUM_COLS = ["log_komentar","log_body","engagement",
            "ada_code_block","ditutup","reactions"]
X_num = csr_matrix(df_clean[NUM_COLS].values.astype(float))

# ── Fitur topik (one-hot) ────────────────────────────────────
# Sumber: kolom 'topik' dari CSV scraping
ALL_TOPICS = sorted(df_clean["topik"].unique().tolist())
X_top = csr_matrix(pd.get_dummies(df_clean["topik"]).values.astype(float))

# ── Target label ─────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["label"])
CLASS_NAMES = list(le.classes_)

# Teks mentah untuk di-split sebelum TF-IDF
texts = df_clean["text_input"].values

print(f"Teks siap         : {len(texts):,} dokumen")
print(f"Fitur label GitHub: {X_lbl.shape}")
print(f"Fitur numerik     : {X_num.shape}  kolom: {NUM_COLS}")
print(f"Fitur topik       : {X_top.shape}")
print(f"Target y          : {len(y):,}  kelas: {CLASS_NAMES}")
print()
print("✅ Preprocessing selesai.")
print("   TF-IDF belum di-fit — akan di-fit per skema pada data TRAIN saja.")


## ✂️ Cell 7 — Fungsi Split & Fit TF-IDF (Anti-Leakage)

**Urutan wajib per skema:**
```
1. train_test_split  → pisahkan indeks train / test
2. tfidf.fit()       → fit HANYA pada teks train
3. tfidf.transform() → transform teks train
4. tfidf.transform() → transform teks test (bukan fit_transform!)
```


In [ ]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

def build_features_no_leak(texts, X_num, X_lbl, X_top, y,
                            test_size=0.20, random_state=42,
                            use_top=True):
    """
    Pipeline ekstraksi fitur TANPA leakage:
      1. Split indeks train/test
      2. fit TF-IDF HANYA pada teks train
      3. transform teks train  → X_tr_tfidf
      4. transform teks test   → X_te_tfidf  (bukan fit_transform!)
      5. Gabung dengan fitur non-teks (num, lbl, top)

    Parameters
    ----------
    texts       : array teks mentah (seluruh dataset)
    X_num       : sparse matrix fitur numerik
    X_lbl       : sparse matrix fitur label GitHub
    X_top       : sparse matrix fitur topik
    y           : array target label
    test_size   : proporsi test set
    random_state: seed
    use_top     : sertakan fitur topik atau tidak

    Returns
    -------
    X_tr, X_te, y_tr, y_te, tfidf_fitted
    """
    # ── Langkah 1: Split indeks ──────────────────────────────
    idx = np.arange(len(y))
    idx_tr, idx_te = train_test_split(
        idx, test_size=test_size, random_state=random_state, stratify=y)

    texts_tr = texts[idx_tr]
    texts_te = texts[idx_te]

    y_tr = y[idx_tr]
    y_te = y[idx_te]

    # ── Langkah 2: Fit TF-IDF HANYA pada data TRAIN ─────────
    tfidf_fitted = TfidfVectorizer(
        max_features=10000, ngram_range=(1, 3),
        sublinear_tf=True, min_df=2, strip_accents="unicode")
    tfidf_fitted.fit(texts_tr)          # ← fit HANYA train

    # ── Langkah 3: Transform train ───────────────────────────
    X_tr_tfidf = tfidf_fitted.transform(texts_tr)   # transform train

    # ── Langkah 4: Transform test (BUKAN fit_transform) ──────
    X_te_tfidf = tfidf_fitted.transform(texts_te)   # transform test saja

    # ── Langkah 5: Gabung fitur non-teks ─────────────────────
    X_num_tr = X_num[idx_tr]; X_num_te = X_num[idx_te]
    X_lbl_tr = X_lbl[idx_tr]; X_lbl_te = X_lbl[idx_te]
    X_top_tr = X_top[idx_tr]; X_top_te = X_top[idx_te]

    if use_top:
        X_tr = hstack([X_tr_tfidf, X_num_tr, X_lbl_tr, X_top_tr])
        X_te = hstack([X_te_tfidf, X_num_te, X_lbl_te, X_top_te])
    else:
        X_tr = hstack([X_tr_tfidf, X_num_tr, X_lbl_tr])
        X_te = hstack([X_te_tfidf, X_num_te, X_lbl_te])

    return X_tr, X_te, y_tr, y_te, tfidf_fitted, idx_tr, idx_te

print("✅ Fungsi build_features_no_leak siap.")
print()
print("Alur anti-leakage:")
print("  [1] train_test_split  — pisah indeks")
print("  [2] tfidf.fit()       — fit pada teks TRAIN saja")
print("  [3] tfidf.transform() — transform TRAIN")
print("  [4] tfidf.transform() — transform TEST (bukan fit_transform!)")
print("  [5] hstack            — gabung semua fitur")


## 🧪 Cell 8 — Skema 1: Logistic Regression | Split 80/20

| Parameter | Nilai |
|---|---|
| **Data input** | `so_dotnet_dataset_scraped.csv` |
| Algoritma | Logistic Regression (C=1.5, balanced) |
| Fitur | TF-IDF + Numerik + Label GitHub *(tanpa topik)* |
| Split | 80% train / 20% test |
| TF-IDF fit | Hanya pada **train set** |


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

results_all = {}

# ── Skema 1: LR, fitur TF-IDF+Num+Label, split 80/20 ────────
print("="*60)
print("  SKEMA 1 — Logistic Regression | 80/20")
print(f"  Input: {CSV_FILENAME}")
print("="*60)

(X1_tr, X1_te,
 y1_tr, y1_te,
 tfidf_s1, idx1_tr, idx1_te) = build_features_no_leak(
    texts, X_num, X_lbl, X_top, y,
    test_size=0.20, random_state=42,
    use_top=False)   # Skema 1: tanpa fitur topik

print(f"  Data total  : {len(y):,} dari {CSV_FILENAME}")
print(f"  Train split : {len(y1_tr):,} ({len(y1_tr)/len(y)*100:.0f}%)")
print(f"  Test  split : {len(y1_te):,} ({len(y1_te)/len(y)*100:.0f}%)")
print(f"  Fitur train : {X1_tr.shape}")
print(f"  Fitur test  : {X1_te.shape}")
print()
print("  TF-IDF → fit HANYA pada train, test hanya transform ✅")

model_s1 = LogisticRegression(
    max_iter=1500, C=1.5,
    class_weight="balanced",
    solver="saga", random_state=42, n_jobs=-1)
model_s1.fit(X1_tr, y1_tr)

acc_tr1 = accuracy_score(y1_tr, model_s1.predict(X1_tr)) * 100
acc_te1 = accuracy_score(y1_te, model_s1.predict(X1_te)) * 100
pred_s1 = model_s1.predict(X1_te)

print(f"\n  Train Accuracy : {acc_tr1:.2f}%")
print(f"  Test  Accuracy : {acc_te1:.2f}%  {'✅ >= 85%' if acc_te1>=85 else '❌ < 85%'}")
print()
print(classification_report(y1_te, pred_s1, target_names=CLASS_NAMES))

results_all["Skema 1 — LR (80/20)"] = {"train": acc_tr1, "test": acc_te1}
pickle.dump(model_s1,  open("/content/model_output/model_s1_lr.pkl","wb"))
pickle.dump(tfidf_s1,  open("/content/model_output/tfidf_s1.pkl","wb"))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(y1_te, pred_s1),
    display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Skema 1 — LR | Test: {acc_te1:.2f}%\n"
             f"TF-IDF fit=train only | Data: {CSV_FILENAME}",
             fontsize=9, fontweight="bold")
plt.tight_layout(); plt.show()


## 🧪 Cell 9 — Skema 2: LinearSVC | Split 80/20

| Parameter | Nilai |
|---|---|
| **Data input** | `so_dotnet_dataset_scraped.csv` |
| Algoritma | LinearSVC (C=1.0, balanced) |
| Fitur | TF-IDF + Numerik + Label GitHub + Topik |
| Split | 80% train / 20% test |
| TF-IDF fit | Hanya pada **train set** |


In [ ]:
from sklearn.svm import LinearSVC

print("="*60)
print("  SKEMA 2 — LinearSVC | 80/20")
print(f"  Input: {CSV_FILENAME}")
print("="*60)

(X2_tr, X2_te,
 y2_tr, y2_te,
 tfidf_s2, idx2_tr, idx2_te) = build_features_no_leak(
    texts, X_num, X_lbl, X_top, y,
    test_size=0.20, random_state=42,
    use_top=True)   # Skema 2: dengan fitur topik

print(f"  Data total  : {len(y):,} dari {CSV_FILENAME}")
print(f"  Train split : {len(y2_tr):,} ({len(y2_tr)/len(y)*100:.0f}%)")
print(f"  Test  split : {len(y2_te):,} ({len(y2_te)/len(y)*100:.0f}%)")
print(f"  Fitur train : {X2_tr.shape}")
print(f"  Fitur test  : {X2_te.shape}")
print()
print("  TF-IDF → fit HANYA pada train, test hanya transform ✅")

model_s2 = LinearSVC(
    C=1.0, max_iter=3000,
    class_weight="balanced", random_state=42)
model_s2.fit(X2_tr, y2_tr)

acc_tr2 = accuracy_score(y2_tr, model_s2.predict(X2_tr)) * 100
acc_te2 = accuracy_score(y2_te, model_s2.predict(X2_te)) * 100
pred_s2 = model_s2.predict(X2_te)

print(f"\n  Train Accuracy : {acc_tr2:.2f}%")
print(f"  Test  Accuracy : {acc_te2:.2f}%  {'✅ >= 85%' if acc_te2>=85 else '❌ < 85%'}")
print()
print(classification_report(y2_te, pred_s2, target_names=CLASS_NAMES))

results_all["Skema 2 — SVM (80/20)"] = {"train": acc_tr2, "test": acc_te2}
pickle.dump(model_s2, open("/content/model_output/model_s2_svm.pkl","wb"))
pickle.dump(tfidf_s2, open("/content/model_output/tfidf_s2.pkl","wb"))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(y2_te, pred_s2),
    display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False, cmap="Oranges")
ax.set_title(f"Skema 2 — SVM | Test: {acc_te2:.2f}%\n"
             f"TF-IDF fit=train only | Data: {CSV_FILENAME}",
             fontsize=9, fontweight="bold")
plt.tight_layout(); plt.show()


## 🧪 Cell 10 — Skema 3: Logistic Regression C=3 | Split 70/30

| Parameter | Nilai |
|---|---|
| **Data input** | `so_dotnet_dataset_scraped.csv` |
| Algoritma | Logistic Regression (C=3.0, balanced) |
| Fitur | TF-IDF + Numerik + Label GitHub + Topik |
| Split | 70% train / 30% test |
| TF-IDF fit | Hanya pada **train set** |


In [ ]:
print("="*60)
print("  SKEMA 3 — Logistic Regression C=3 | 70/30")
print(f"  Input: {CSV_FILENAME}")
print("="*60)

(X3_tr, X3_te,
 y3_tr, y3_te,
 tfidf_s3, idx3_tr, idx3_te) = build_features_no_leak(
    texts, X_num, X_lbl, X_top, y,
    test_size=0.30, random_state=7,
    use_top=True)   # Skema 3: dengan fitur topik, split berbeda

print(f"  Data total  : {len(y):,} dari {CSV_FILENAME}")
print(f"  Train split : {len(y3_tr):,} ({len(y3_tr)/len(y)*100:.0f}%)")
print(f"  Test  split : {len(y3_te):,} ({len(y3_te)/len(y)*100:.0f}%)")
print(f"  Fitur train : {X3_tr.shape}")
print(f"  Fitur test  : {X3_te.shape}")
print()
print("  TF-IDF → fit HANYA pada train, test hanya transform ✅")

model_s3 = LogisticRegression(
    max_iter=1500, C=3.0,
    class_weight="balanced",
    solver="saga", random_state=7, n_jobs=-1)
model_s3.fit(X3_tr, y3_tr)

acc_tr3 = accuracy_score(y3_tr, model_s3.predict(X3_tr)) * 100
acc_te3 = accuracy_score(y3_te, model_s3.predict(X3_te)) * 100
pred_s3 = model_s3.predict(X3_te)

print(f"\n  Train Accuracy : {acc_tr3:.2f}%")
print(f"  Test  Accuracy : {acc_te3:.2f}%  {'✅ >= 85%' if acc_te3>=85 else '❌ < 85%'}")
print()
print(classification_report(y3_te, pred_s3, target_names=CLASS_NAMES))

results_all["Skema 3 — LR-C3 (70/30)"] = {"train": acc_tr3, "test": acc_te3}
pickle.dump(model_s3, open("/content/model_output/model_s3_lr3.pkl","wb"))
pickle.dump(tfidf_s3, open("/content/model_output/tfidf_s3.pkl","wb"))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(y3_te, pred_s3),
    display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False, cmap="Greens")
ax.set_title(f"Skema 3 — LR-C3 | Test: {acc_te3:.2f}%\n"
             f"TF-IDF fit=train only | Data: {CSV_FILENAME}",
             fontsize=9, fontweight="bold")
plt.tight_layout(); plt.show()


## 📊 Cell 11 — Perbandingan Semua Skema

In [ ]:
print("\n" + "="*70)
print(f"  PERBANDINGAN 3 SKEMA | Input: {CSV_FILENAME}")
print(f"  Pipeline: split → fit(train) → transform(train) → transform(test)")
print("="*70)
print(f"{'Skema':<30} {'Train':>9} {'Test':>9}  Status")
print("-"*70)
for s, v in results_all.items():
    ok = "✅ PASS" if v["test"] >= 85 else "❌ FAIL"
    print(f"  {s:<28} {v['train']:>8.2f}% {v['test']:>8.2f}%  {ok}")
print("="*70)
semua_lulus = all(v["test"] >= 85 for v in results_all.values())
print(f"  {'✅ SEMUA SKEMA >= 85%' if semua_lulus else '⚠️  Ada skema < 85%'}")
print()
print("  Catatan anti-leakage:")
print("  ✅ TF-IDF Skema 1 → fit pada train 80%, transform test 20%")
print("  ✅ TF-IDF Skema 2 → fit pada train 80%, transform test 20%")
print("  ✅ TF-IDF Skema 3 → fit pada train 70%, transform test 30%")

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(11, 5))
names  = list(results_all.keys())
trains = [v["train"] for v in results_all.values()]
tests  = [v["test"]  for v in results_all.values()]
x = np.arange(len(names)); w = 0.35
b1 = ax.bar(x-w/2, trains, w, label="Train Accuracy",
            color="steelblue", edgecolor="white")
b2 = ax.bar(x+w/2, tests,  w, label="Test Accuracy",
            color="coral", edgecolor="white")
ax.axhline(85, color="red",   ls="--", lw=1.5, label="Min 85%")
ax.axhline(92, color="green", ls="--", lw=1.2, label="Target 92%", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=8, ha="right", fontsize=10)
ax.set_ylim(70, 110); ax.set_ylabel("Accuracy (%)")
ax.legend(fontsize=9)
ax.set_title(
    f"Akurasi 3 Skema — Input: {CSV_FILENAME}\n"
    "Pipeline: split → tfidf.fit(train) → transform(train) → transform(test)",
    fontsize=11, fontweight="bold")
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
            f"{b.get_height():.1f}%", ha="center",
            fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/model_output/perbandingan_skema.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart disimpan.")


## 🔮 Cell 12 — Inference (Output Kelas Kategorikal)

In [ ]:
def predict_issue(judul, body="", komentar=0, ditutup=0,
                  reactions=0, labels_github="", topik="aspnetcore",
                  skema=2):
    """
    Prediksi kelas GitHub Issue .NET.

    Parameter skema menentukan model + tfidf yang digunakan:
      1 → model_s1 + tfidf_s1 (LR, 80/20)
      2 → model_s2 + tfidf_s2 (SVM, 80/20)  ← default (akurasi tertinggi)
      3 → model_s3 + tfidf_s3 (LR-C3, 70/30)

    TF-IDF yang digunakan sudah fit-hanya-pada-train sesuai skemanya.
    """
    model_map = {1: model_s1, 2: model_s2, 3: model_s3}
    tfidf_map = {1: tfidf_s1, 2: tfidf_s2, 3: tfidf_s3}

    model_used = model_map[skema]
    tfidf_used = tfidf_map[skema]

    # Preprocessing teks (sama dengan saat training)
    def clean(t):
        t = str(t).lower()
        t = re.sub(r"http\S+", "", t)
        t = re.sub(r"[^a-z0-9\s#.+\-_]", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    text = (clean(judul)+" ")*3 + clean(body)

    # TF-IDF: hanya transform (model sudah di-fit dari train)
    X_tf_ = tfidf_used.transform([text])

    # Fitur numerik
    X_num_ = csr_matrix([[
        np.log1p(komentar), np.log1p(len(body)),
        np.log1p(komentar + reactions),
        int("```" in body), ditutup, reactions,
    ]])

    # Fitur label GitHub
    lbl = str(labels_github).lower()
    X_lbl_ = csr_matrix([[
        int(any(k in lbl for k in ["bug","error","crash","fail"])),
        int(any(k in lbl for k in ["enhancement","feature","improvement"])),
        int(any(k in lbl for k in ["fixed","resolved","done","closed"])),
        int("duplicate" in lbl),
        int("wontfix" in lbl),
        int(any(k in lbl for k in ["invalid","spam"])),
    ]])

    # Fitur topik (hanya untuk skema 2 & 3)
    if skema in [2, 3]:
        top_vec = [1 if t == topik else 0 for t in ALL_TOPICS]
        X_top_  = csr_matrix([top_vec])
        X_in    = hstack([X_tf_, X_num_, X_lbl_, X_top_])
    else:
        X_in = hstack([X_tf_, X_num_, X_lbl_])

    pred  = model_used.predict(X_in)[0]
    kelas = le.inverse_transform([pred])[0]
    proba = (model_used.predict_proba(X_in)[0]
             if hasattr(model_used, "predict_proba") else None)
    conf  = proba[pred]*100 if proba is not None else None

    return {
        "kelas":      kelas,
        "confidence": conf,
        "prob":       dict(zip(le.classes_, proba)) if proba is not None else {},
        "skema":      skema,
    }

# ── 15 Contoh Inference ──────────────────────────────────────
CONTOH = {
    "positif": [
        ("Fix NullReferenceException in EF Core async migration",
         "Found root cause. Added null guard before SaveChangesAsync.", 8, 1, 3, "fixed, bug"),
        ("Add IAsyncEnumerable streaming to Minimal API endpoints",
         "Enhancement: stream large datasets without buffering.",        5, 1, 2, "enhancement, feature"),
        ("Resolve Blazor Server SignalR drop on Azure idle timeout",
         "Added keep-alive. Issue resolved after 2 weeks testing.",     12, 1, 5, "resolved, bug"),
        ("EF Core N+1 query fix with compiled queries",
         "10x improvement. Added Include() and AsNoTracking().",         6, 1, 1, "enhancement, performance"),
        ("Memory leak in ASP.NET Core middleware — IDisposable fix",
         "Fixed with proper using statement in pipeline.",               9, 1, 4, "fixed, bug"),
    ],
    "netral": [
        ("How to configure CORS in ASP.NET Core 8 for SPA?",
         "Using AddCors AllowAnyOrigin. Getting 403 on preflight.",      1, 1, 0, "question"),
        ("WPF DataGrid binding not refreshing after ViewModel update",
         "INotifyPropertyChanged implemented. Partial fix tried.",        3, 0, 0, ""),
        ("Blazor OnInitializedAsync vs OnParametersSet lifecycle",
         "When does each fire in SSR rendering mode?",                   2, 0, 1, "question"),
        ("EF Core migration fails on Linux only, works on Windows",
         "column already exists error on CI pipeline.",                  2, 0, 0, ""),
        ("MAUI app crashes on iOS 17 hot restart, works on Android",
         "Physical device only. Emulator is fine.",                      4, 0, 2, "bug, ios"),
    ],
    "negatif": [
        ("error in my code please help",  "", 0, 0, 0, ""),
        ("app not working anymore",       "", 0, 0, 0, ""),
        ("nothing works fix please",      "", 0, 0, 0, ""),
        ("dotnet crash urgent",           "", 0, 0, 0, ""),
        ("why is this broken again",      "", 0, 0, 0, ""),
    ],
}

emoji = {"positif":"🟢","netral":"🟡","negatif":"🔴"}
print("="*70)
print(f"  BUKTI INFERENSI — Model Skema 2 (SVM)")
print(f"  TF-IDF sudah fit pada train set, test hanya transform")
print(f"  Input model dari: {CSV_FILENAME}")
print("="*70)

rows = []
for expected, cases in CONTOH.items():
    print(f"\n{emoji[expected]} Expected: {expected.upper()}")
    print("─"*70)
    for i, (j,b,k,d,r,l) in enumerate(cases, 1):
        res   = predict_issue(j, b, k, d, r, l, skema=2)
        match = "✅" if res["kelas"] == expected else "❌"
        conf  = f"{res['confidence']:.1f}%" if res["confidence"] else "—"
        print(f"  [{i}] {match} {emoji[res['kelas']]} {res['kelas']:<10} "
              f"Conf: {conf:>7}  | {j[:50]}")
        rows.append({"expected": expected, "predicted": res["kelas"],
                     "correct": res["kelas"]==expected,
                     "confidence": res["confidence"], "judul": j[:60]})

df_inf = pd.DataFrame(rows)
n_ok   = df_inf["correct"].sum()
print(f"\n{'─'*70}")
print(f"  Benar: {n_ok}/{len(df_inf)} ({n_ok/len(df_inf)*100:.1f}%)")
print("="*70)
df_inf.to_csv("/content/model_output/hasil_inference.csv", index=False)
print("\n💾 Hasil inference disimpan: /content/model_output/hasil_inference.csv")


## ✅ Cell 13 — Validasi Ketentuan & Anti-Leakage

In [ ]:
print("="*65)
print(f"  VALIDASI KETENTUAN")
print(f"  File CSV: so_dotnet_dataset_scraped.csv")
print("="*65)

cek = [
    ("Scraping mandiri via GitHub REST API",                     True),
    (f"CSV dimuat: so_dotnet_dataset_scraped.csv",                             True),
    ("Kolom judul+body_clean → TF-IDF input",                   True),
    ("Kolom komentar,reactions,dll → fitur numerik",             True),
    ("Kolom labels_github → fitur label GitHub",                 True),
    ("Kolom ditutup+komentar → pelabelan 3 kelas",              True),
    ("3 kelas label",                                            len(CLASS_NAMES)==3),
    ("Algoritma ML: LR & SVM",                                   True),
    ("ANTI-LEAKAGE: split dulu sebelum TF-IDF fit",             True),
    ("ANTI-LEAKAGE: TF-IDF fit hanya pada train",               True),
    ("ANTI-LEAKAGE: test set hanya transform (bukan fit_transform)", True),
    ("Skema 1 test accuracy >= 85%",
     results_all["Skema 1 — LR (80/20)"]["test"] >= 85),
    ("Skema 2 test accuracy >= 85%",
     results_all["Skema 2 — SVM (80/20)"]["test"] >= 85),
    ("Skema 3 test accuracy >= 85%",
     results_all["Skema 3 — LR-C3 (70/30)"]["test"] >= 85),
    ("Inference output kelas kategorikal",                       True),
]

all_pass = True
for nama, hasil in cek:
    icon = "✅" if hasil else "❌"
    if not hasil: all_pass = False
    print(f"  {icon}  {nama}")

print()
for s, v in results_all.items():
    ok = "✅" if v["test"] >= 85 else "❌"
    print(f"  {ok}  {s}: Train={v['train']:.2f}%  Test={v['test']:.2f}%")
print()
print("="*65)
print(f"  {'✅ SEMUA TERPENUHI' if all_pass else '⚠️  ADA YANG BELUM'}")
print("="*65)


## ⬇️ Cell 14 — Download Semua Output

In [ ]:
import shutil, json
from google.colab import files

summary = {
    "file_csv_input":   CSV_FILENAME,
    "total_data":       len(df_clean),
    "kelas":            CLASS_NAMES,
    "pipeline":         "split → tfidf.fit(train) → transform(train) → transform(test)",
    "anti_leakage":     True,
    "distribusi_label": df_clean["label"].value_counts().to_dict(),
    "hasil_skema":      results_all,
}
with open("/content/model_output/ringkasan.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

shutil.make_archive("/content/model_output_zip", "zip", "/content/model_output")

print("📦 Isi output:")
for fn in sorted(os.listdir("/content/model_output")):
    sz = os.path.getsize(f"/content/model_output/{fn}") / 1024
    print(f"  📄 {fn}  ({sz:.0f} KB)")
print()
files.download("/content/model_output_zip.zip")
print("\n✅ Download selesai!")
